In [ ]:
adaptive RAG

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import Docx2txtLoader, PyPDFLoader, TextLoader, CSVLoader, JSONLoader, DirectoryLoader

file_path = "./North_Korean_Tactics.pdf"
save_vector = "./north_korean_tactics_faiss"

def get_loader(filename):
    loader_classes = {
        'docx': Docx2txtLoader,
        'pdf': PyPDFLoader,
        'txt': TextLoader,
        'csv': CSVLoader,
        'json': JSONLoader,
        # 'hwp': HWPLoader
    }

    _, file_extension = os.path.splitext(filename)
    file_extension = file_extension.lstrip('.')

    loader_class = loader_classes.get(file_extension)

    if not loader_class:
        raise ValueError(f"No loader available for file extension '{file_extension}'")
    
    # JSON 파일인 경우 jq_schema 인자를 추가하여 반환
    if file_extension == 'json':
        return loader_class(filename, jq_schema='.', text_content=False)
    else:
        # 그 외의 경우 일반적인 방식으로 로더 반환
        return loader_class(filename)

loader = get_loader(file_path)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500, chunk_overlap=0
)

doc_splits = text_splitter.split_documents(loader.load())

# Add to vectorDB
# vectorstore = Chroma.from_documents(
#     documents=doc_splits,
#     collection_name="rag-chroma",
#     embedding=OpenAIEmbeddings(),
# )

vectorstore = FAISS.from_documents(doc_splits, OpenAIEmbeddings())

vectorstore.save_local(save_vector)

retriever = vectorstore.as_retriever()

In [ ]:
### Retrieval Grader


# Data model
class GradeDocuments(BaseModel):
    """Binary score for relevance check on retrieved documents."""

    binary_score: str = Field(
        description="Documents are relevant to the question, 'yes' or 'no'"
    )


# LLM with function call
llm = ChatOpenAI(
        api_key="ai",
        model="openai/gpt-oss-20b",
        base_url="http://192.168.0.110:8000/v1",
        temperature=0,
    )
structured_llm_grader = llm.with_structured_output(GradeDocuments)

# Prompt
system = """You are a grader assessing relevance of a retrieved document to a user question. \n 
    If the document contains keyword(s) or semantic meaning related to the user question, grade it as relevant. \n
    It does not need to be a stringent test. The goal is to filter out erroneous retrievals. \n
    Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question."""
grade_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Retrieved document: \n\n {document} \n\n User question: {question}"),
    ]
)

retrieval_grader = grade_prompt | structured_llm_grader
question = "agent memory"
# docs = retriever.get_relevant_documents(question)
docs = retriever.invoke(question)
doc_txt = docs[1].page_content

In [ ]:
### Generate

from langchain_classic import hub
from langchain_core.output_parsers import StrOutputParser

# Prompt
# prompt = hub.pull("rlm/rag-prompt")

pull = """
    You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
"""
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", pull),
        ("human", "Question: {question}, Context: {context} "),
    ]
)


# LLM
llm = ChatOpenAI(
        api_key="ai",
        model="openai/gpt-oss-20b",
        base_url="http://192.168.0.110:8000/v1",
        temperature=0,
    )


# Post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# Chain
rag_chain = prompt | llm | StrOutputParser()

# Run
generation = rag_chain.invoke({"context": docs, "question": question})


In [ ]:
### Hallucination Grader


# Data model
class GradeHallucinations(BaseModel):
    """Binary score for hallucination present in generation answer."""

    binary_score: str = Field(
        description="Answer is grounded in the facts, 'yes' or 'no'"
    )


# LLM with function call
llm = ChatOpenAI(
        api_key="ai",
        model="openai/gpt-oss-20b",
        base_url="http://192.168.0.110:8000/v1",
        temperature=0,
    )
structured_llm_grader = llm.with_structured_output(GradeHallucinations)

# Prompt
system = """You are a grader assessing whether an LLM generation is grounded in / supported by a set of retrieved facts. \n 
     Give a binary score 'yes' or 'no'. 'Yes' means that the answer is grounded in / supported by the set of facts."""
hallucination_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Set of facts: \n\n {documents} \n\n LLM generation: {generation}"),
    ]
)

hallucination_grader = hallucination_prompt | structured_llm_grader
hallucination_grader.invoke({"documents": docs, "generation": generation})

In [ ]:
### Answer Grader


# Data model
class GradeAnswer(BaseModel):
    """Binary score to assess answer addresses question."""

    binary_score: str = Field(
        description="Answer addresses the question, 'yes' or 'no'"
    )


# LLM with function call
llm = ChatOpenAI(
        api_key="ai",
        model="openai/gpt-oss-20b",
        base_url="http://192.168.0.110:8000/v1",
        temperature=0,
    )
structured_llm_grader = llm.with_structured_output(GradeAnswer)

# Prompt
system = """You are a grader assessing whether an answer addresses / resolves a question \n 
     Give a binary score 'yes' or 'no'. Yes' means that the answer resolves the question."""
answer_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "User question: \n\n {question} \n\n LLM generation: {generation}"),
    ]
)

answer_grader = answer_prompt | structured_llm_grader
answer_grader.invoke({"question": question, "generation": generation})

In [ ]:
### Question Re-writer

# LLM
llm = ChatOpenAI(
        api_key="ai",
        model="openai/gpt-oss-20b",
        base_url="http://192.168.0.110:8000/v1",
        temperature=0,
    )

# Prompt
system = """You a question re-writer that converts an input question to a better version that is optimized \n 
     for vectorstore retrieval. Look at the input and try to reason about the underlying semantic intent / meaning."""
re_write_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        (
            "human",
            "Here is the initial question: \n\n {question} \n Formulate an improved question.",
        ),
    ]
)

question_rewriter = re_write_prompt | llm | StrOutputParser()
question_rewriter.invoke({"question": question})